In [1]:
# !pip install mlflow boto3 awscli optuna xgboost imbalanced-learn

In [2]:
# !aws configure

In [3]:
# !aws configure

In [4]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000")

d:\MLops\MLops-Youtube-Sentiment-Insights\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Set or create an experiment
mlflow.set_experiment("Experiment 5 - ML Algo with HP Tuning")

<Experiment: artifact_location='s3://yoosuf520-mlflow-s3/12', creation_time=1782288359078, effective_trace_archival_retention=None, experiment_id='12', last_update_time=1782288359078, lifecycle_stage='active', name='Experiment 5 - ML Algo with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [6]:
import optuna
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [7]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [8]:
import mlflow
import mlflow.xgboost
import mlflow.sklearn
import gc
import optuna
from xgboost import XGBClassifier
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# --- Set Remote Tracking Server (From your previous steps) ---
mlflow.set_tracking_uri("http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000")
mlflow.set_experiment("Experiment 5 - ML Algo with HP Tuning")

# Step 1: Remap class labels
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})
df = df.dropna(subset=['category'])

# Step 2: TF-IDF settings
ngram_range = (1, 3)
max_features = 3000

# Step 3: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'], df['category'],
    test_size=0.2, random_state=42, stratify=df['category']
)

# Step 4: Vectorization
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Step 5: Resampling (Note: Code mentions SMOTE in tags, but uses RandomOverSampler here. Both work!)
ros = RandomOverSampler(random_state=42)
X_train_vec, y_train = ros.fit_resample(X_train_vec, y_train)
gc.collect()


# Step 6: log_mlflow — Fixed with nested=True to keep MLflow dashboard clean
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):
    # ADDED: nested=True ensures these trials stay grouped under the parent run
    with mlflow.start_run(nested=True):
        # tags
        mlflow.set_tag(
            "mlflow.runName",
            f"Trial_{trial_number}_{model_name}_ROS_TFIDF_Trigrams"
        )
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.log_param("algo_name", model_name)

        # log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # train and predict
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # log classification report
        classification_rep = classification_report(
            y_test, y_pred, output_dict=True
        )
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # log model using correct flavor
        if isinstance(model, XGBClassifier):
            mlflow.xgboost.log_model(model, f"{model_name}_model")  
        else:
            mlflow.sklearn.log_model(model, f"{model_name}_model")  

        return accuracy  


# Step 7: Optuna objective
def objective_xgboost(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)

    params = {
        'n_estimators': n_estimators,
        'learning_rate': learning_rate,
        'max_depth': max_depth
    }

    model = XGBClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        random_state=42,
        eval_metric='mlogloss'
    )

    accuracy = log_mlflow(
        "XGBoost", model,
        X_train_vec, X_test_vec,
        y_train, y_test,
        params, trial.number    
    )
    return accuracy


# Step 8: Run Optuna
def run_optuna_experiment():
    # ADDED: Top-level parent run to encapsulate the entire study nicely
    with mlflow.start_run(run_name="Optuna_XGBoost_Tuning_Session"):
        
        study = optuna.create_study(direction="maximize")
        study.optimize(objective_xgboost, n_trials=20)

        best_params = study.best_params
        best_model = XGBClassifier(
            n_estimators=best_params['n_estimators'],
            learning_rate=best_params['learning_rate'],
            max_depth=best_params['max_depth'],
            random_state=42,
            eval_metric='mlogloss'
        )

        # log best model under the parent run
        log_mlflow(
            "XGBoost", best_model,
            X_train_vec, X_test_vec,
            y_train, y_test,
            best_params, "Best"     
        )

        print("\n--- Optimization Finished Successfully! ---")
        # plots (Will open in interactive windows if using notebooks/GUI)
        try:
            optuna.visualization.plot_param_importances(study).show()
            optuna.visualization.plot_optimization_history(study).show()
        except Exception as e:
            print(f"Plotting skipped or headlessly run: {e}")

# Run it
run_optuna_experiment()

[I 2026-06-24 16:04:19,428] A new study created in memory with name: no-name-28e7129f-9df6-4a01-9864-45d0bf68ca9f
2026/06/24 16:07:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/24 16:08:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run Trial_0_XGBoost_ROS_TFIDF_Trigrams at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12/runs/116b3f9e080945b0bfa9af6e42b173dd
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12


[I 2026-06-24 16:08:32,389] Trial 0 finished with value: 0.5401609164052912 and parameters: {'n_estimators': 200, 'learning_rate': 0.0013695168639313443, 'max_depth': 4}. Best is trial 0 with value: 0.5401609164052912.
2026/06/24 16:15:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/24 16:16:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run Trial_1_XGBoost_ROS_TFIDF_Trigrams at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12/runs/128538fb69634e3fa88f9153a39ccc89
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12


[I 2026-06-24 16:17:08,511] Trial 1 finished with value: 0.7130778671757807 and parameters: {'n_estimators': 277, 'learning_rate': 0.024015132943834193, 'max_depth': 6}. Best is trial 1 with value: 0.7130778671757807.
2026/06/24 16:21:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/24 16:21:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

🏃 View run Trial_8_XGBoost_ROS_TFIDF_Trigrams at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12/runs/6bb8b02056484ad7b21c2e1ad48e92fc
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12


[I 2026-06-24 17:04:35,927] Trial 8 finished with value: 0.7331242329196782 and parameters: {'n_estimators': 210, 'learning_rate': 0.037756842012056896, 'max_depth': 7}. Best is trial 7 with value: 0.7620346379380881.
2026/06/24 17:05:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/24 17:05:58 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run Trial_9_XGBoost_ROS_TFIDF_Trigrams at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12/runs/213a765a1a864150a6f1e07356277ae3
🧪 View experiment at: http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/#/experiments/12


[I 2026-06-24 17:06:05,923] Trial 9 finished with value: 0.4951588708577663 and parameters: {'n_estimators': 127, 'learning_rate': 0.000271312668455171, 'max_depth': 3}. Best is trial 7 with value: 0.7620346379380881.
[W 2026-06-24 17:17:39,977] Trial 10 failed with parameters: {'n_estimators': 52, 'learning_rate': 0.00021182251401779577, 'max_depth': 10} because of the following error: MlflowException('API request to http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/api/2.0/mlflow/runs/get failed with exception HTTPConnectionPool(host=\'ec2-43-205-213-111.ap-south-1.compute.amazonaws.com\', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/get?run_uuid=60d86ae9d7ce410c95c7009a9f3d8fa6&run_id=60d86ae9d7ce410c95c7009a9f3d8fa6 (Caused by NameResolutionError("HTTPConnection(host=\'ec2-43-205-213-111.ap-south-1.compute.amazonaws.com\', port=5000): Failed to resolve \'ec2-43-205-213-111.ap-south-1.compute.amazonaws.com\' ([Errno 11001] getaddrinfo failed)"))').
T

MlflowException: API request to http://ec2-43-205-213-111.ap-south-1.compute.amazonaws.com:5000/api/2.0/mlflow/runs/get failed with exception HTTPConnectionPool(host='ec2-43-205-213-111.ap-south-1.compute.amazonaws.com', port=5000): Max retries exceeded with url: /api/2.0/mlflow/runs/get?run_uuid=77945c3386fb4d5d8febd1f3b87fe6e8&run_id=77945c3386fb4d5d8febd1f3b87fe6e8 (Caused by NameResolutionError("HTTPConnection(host='ec2-43-205-213-111.ap-south-1.compute.amazonaws.com', port=5000): Failed to resolve 'ec2-43-205-213-111.ap-south-1.compute.amazonaws.com' ([Errno 11001] getaddrinfo failed)"))